In [3]:
import os
from pathlib import Path
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("EV_Peak_Load_Forecasting") \
    .getOrCreate()

# Base directory path
base_dir = r"C:\Data_from_shahana_onedrive\CIMT\FinalCapstoneProject\Project_files\DataSets\evwatts.public\evwatts.public"

# 1. Verify directory exists and list files
if not os.path.exists(base_dir):
    print(f"❌ Directory does not exist: {base_dir}")
else:
    print("📁 Files found in folder:")
    all_files = os.listdir(base_dir)
    for f in all_files:
        print(f" - {f}")

    # 2. Automatically locate the EVSE file regardless of exact extension
    evse_filename = next((f for f in all_files if "evse" in f.lower()), None)

    if evse_filename:
        full_path = os.path.join(base_dir, evse_filename)
        print(f"\n✅ Target file located: {full_path}")

        # 3. Use pathlib to generate a clean Windows file URI
        file_uri = Path(full_path).as_uri()

        # 4. Read into PySpark
        df_evse = spark.read.csv(file_uri, header=True, inferSchema=True)
        
        print("\n--- EVSE Data Preview ---")
        df_evse.show(5)

        # 5. Extract location columns
        location_cols = [
            c for c in df_evse.columns 
            if any(term in c.lower() for term in ['zip', 'state', 'lat', 'lon', 'city', 'region'])
        ]
        
        print("Found Location Columns:", location_cols)
        if location_cols:
            df_evse.select(location_cols).distinct().show(10, truncate=False)
    else:
        print("\n❌ Could not find an 'evse' file in the directory!")

📁 Files found in folder:
 - evwatts.public.connector.csv
 - evwatts.public.dictionary.txt
 - evwatts.public.evse.csv
 - evwatts.public.session.csv
 - evwatts.public.vehicles.csv
 - evwatts.public.vehiclesessions.csv
 - evwatts.public.vehicletrips.csv
 - ev_weather_processed.parquet
 - weather_data.csv

✅ Target file located: C:\Data_from_shahana_onedrive\CIMT\FinalCapstoneProject\Project_files\DataSets\evwatts.public\evwatts.public\evwatts.public.evse.csv

--- EVSE Data Preview ---
+-------+--------------------+----------+--------+---------+------------+------------+-------+
|evse_id|          metro_area|  land_use|  region|num_ports|charge_level|       venue|pricing|
+-------+--------------------+----------+--------+---------+------------+------------+-------+
|   6034|        Undesignated|Metro Area|Mountain|        1|        DCFC|    Corridor|   Paid|
|   6065|        Undesignated|Metro Area|Mountain|        1|        DCFC|    Corridor|   Paid|
|   6100|Phoenix-Mesa-Chan...|Metro Ar

In [4]:
# See the top metro areas in your EVSE dataset (excluding Undesignated)
df_evse.filter(df_evse.metro_area != "Undesignated") \
       .groupBy("metro_area") \
       .count() \
       .orderBy("count", ascending=False) \
       .show(10, truncate=False)

+-------------------------------------------------------+-----+
|metro_area                                             |count|
+-------------------------------------------------------+-----+
|Portland-Vancouver-Hillsboro, OR-WA Metro Area         |2939 |
|Washington-Arlington-Alexandria, DC-VA-MD-WV Metro Area|2189 |
|Baltimore-Columbia-Towson, MD Metro Area               |1946 |
|Detroit-Warren-Dearborn, MI Metro Area                 |1843 |
|Philadelphia-Camden-Wilmington, PA-NJ-DE-MD Metro Area |1603 |
|New York-Newark-Jersey City, NY-NJ-PA Metro Area       |1166 |
|Burlington-South Burlington, VT Metro Area             |1163 |
|Las Vegas-Henderson-Paradise, NV Metro Area            |1027 |
|Boston-Cambridge-Newton, MA-NH Metro Area              |997  |
|Kansas City, MO-KS Metro Area                          |760  |
+-------------------------------------------------------+-----+
only showing top 10 rows


In [5]:
from pathlib import Path
from pyspark.sql.functions import min as spark_min, max as spark_max

# Path to session file
session_path = Path(r"C:\Data_from_shahana_onedrive\CIMT\FinalCapstoneProject\Project_files\DataSets\evwatts.public\evwatts.public\evwatts.public.session.csv").as_uri()

# Read session data
df_session = spark.read.csv(session_path, header=True, inferSchema=True)

# Find date range (replace 'connect_time' if your timestamp column has a different name)
time_col = [c for c in df_session.columns if 'time' in c.lower() or 'date' in c.lower()][0]
print(f"Using timestamp column: {time_col}")

date_bounds = df_session.select(
    spark_min(time_col).cast("string"), 
    spark_max(time_col).cast("string")
).collect()[0]

print(f"📅 Start Date: {date_bounds[0]}")
print(f"📅 End Date:   {date_bounds[1]}")

Using timestamp column: start_datetime
📅 Start Date: 2019-06-25 14:32:34
📅 End Date:   2022-12-31 23:58:58


In [6]:
import requests

# Portland Metro Coordinates (Top metro area in your dataset)
LATITUDE = 45.5152
LONGITUDE = -122.6784

# Extracted exact dates from session.csv
START_DATE = "2019-06-25"
END_DATE = "2022-12-31"

# Open-Meteo Historical Weather API URL
url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={LATITUDE}&longitude={LONGITUDE}&"
    f"start_date={START_DATE}&end_date={END_DATE}&"
    f"hourly=temperature_2m,relative_humidity_2m,precipitation,direct_radiation&"
    f"format=csv"
)

# Target save directory matching your EV Watts dataset location
output_path = r"C:\Data_from_shahana_onedrive\CIMT\FinalCapstoneProject\Project_files\DataSets\evwatts.public\evwatts.public\weather_data.csv"

print("Downloading weather data from Open-Meteo...")
response = requests.get(url)

if response.status_code == 200:
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(response.text)
    print("✅ Weather dataset downloaded successfully as weather_data.csv!")
else:
    print(f"❌ Download failed with status code: {response.status_code}")

✅ Weather dataset downloaded successfully as weather_data.csv!
